# v0.3: Evaluation Under Clustered Input Distribution

## Motivation

In the previous experiment (v0.2), both Local Fix and Suffix Flip showed similar average performance under uniformly random binary sequences.

This result suggests that when the input sequence has no structural correlation, neither strategy can gain significant advantage.

However, real-world card states may not always follow a completely random distribution. 
Before sufficient shuffling, similar states may appear in clusters due to previous usage patterns or physical arrangement.

Therefore, this version introduces a correlated binary sequence model to investigate whether input structure affects the performance of different strategies.

## Research Question

Does the existence of local state correlation provide advantages for Suffix Flip compared with Local Fix?

## Experimental Change

Compared with v0.2:

- Algorithms remain unchanged.
- Only the sequence generation method is modified.
- A clustered binary sequence generator is introduced.

The experiment compares strategy performance under:

1. Independent random sequences (IID Bernoulli distribution)
2. Clustered sequences generated with state correlation

## Import & Reuse Functions

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statistics import mean
from scipy.stats import linregress
import sys
sys.path.insert(0, '..')
from src.algorithms import local_fix, Suffix_flip
from src.generators import clustered_sequence_generator


## Experiment Parameters

In [ ]:
sequence_length = 50
N_simulation = 10000
random.seed(42)

## Clustered Generator

## Monte Carlo Simulation


In [ ]:
p_values = np.arange(0.5,0.96,0.01)
experiment_results = []
for p in p_values:
    local_results = []
    suffix_results = []
    for experiment in range(N_simulation):
        sequence = clustered_sequence_generator(sequence_length,p)
        sequence_local = sequence.copy()
        sequence_suffix = sequence.copy()
        _,operations_local = local_fix(sequence_local)
        local_results.append(operations_local)
        _,operations_suffix = Suffix_flip(sequence_suffix)
        suffix_results.append(operations_suffix)
    local_mean = mean(local_results)
    suffix_mean = mean(suffix_results)
    delta = local_mean - suffix_mean
    experiment_results.append(
        [
            p,
            local_mean,
            suffix_mean,
            delta,
            min(local_results),
            max(local_results),
            min(suffix_results),
            max(suffix_results)
        ]
    )
df = pd.DataFrame(
    experiment_results,
    columns=[
        "p",
        "local_mean",
        "suffix_mean",
        "delta",
        "local_min",
        "local_max",
        "suffix_min",
        "suffix_max"
    ]
)
print(df)
df.describe()

## Visualization


In [ ]:
#delta-p
plt.scatter(
    df["p"],
    df["delta"]
)

plt.xlabel("p")
plt.ylabel("Delta")
plt.title("Delta vs p")

plt.show()

#exploratory analysis
df["cluster_length"] = 1 / (1 - df["p"])

plt.scatter(
    df["cluster_length"],
    df["delta"]
)

plt.xlabel("Expected Cluster Length")
plt.ylabel("Delta")
plt.title("Delta vs Expected Cluster Length")

plt.show()

## Regression Analysis

In [ ]:
x=df["p"]
y=df["delta"]

regression=linregress(x,y)
print("===== Linear Regression: Delta vs p_same =====")

print(f"Slope: {regression.slope:.4f}")

print(f"Intercept: {regression.intercept:.4f}")

print(f"R-squared: {regression.rvalue**2:.4f}")

print(f"P-value: {regression.pvalue:.6f}")
#results
plt.scatter(
    df["p"],
    df["delta"],
    label="Simulation"
)

x = df["p"]

y_pred = regression.intercept + regression.slope*x

plt.plot(
    x,
    y_pred,
    label="Linear Fit"
)

plt.xlabel("p")
plt.ylabel("Delta")
plt.legend()

plt.show()
df.to_csv(
    "v0.3_cluster_experiment_results.csv",
    index=False
)

## Conclusion

In this stage, we investigated the performance difference between **Local Fix** and **Suffix Flip** under clustered input sequences.

By introducing a clustering probability parameter $p$, we controlled the correlation structure of generated sequences and conducted Monte Carlo simulations across different values of $p$.

The main findings are summarized as follows:

---

### 1. Suffix Flip benefits from clustered input structures

Under random-like sequences ($p \approx 0.5$), Local Fix and Suffix Flip show similar performance, with the average operation difference approaching zero.

As the clustering probability increases, Suffix Flip requires fewer operations compared with Local Fix, indicating that Suffix Flip can better exploit local structural information in the input sequence.

---

### 2. Performance gap increases approximately linearly with clustering probability

We define the performance gap as:

$$
\Delta =
E(Operations_{Local})-
E(Operations_{Suffix})
$$

Monte Carlo experiments show that the performance gap has a strong approximately linear relationship with the clustering probability:

$$
\Delta=f(p)
$$

A linear regression model was fitted:

$$
\Delta=\beta_0+\beta_1p+\epsilon
$$

The fitted model is:

$$
\Delta=48.8958p-24.4336
$$

with:

$$
R^2=0.9999
$$

within the tested parameter range.

This indicates that clustering probability is a strong predictor of the relative advantage of Suffix Flip over Local Fix.

---

### 3. Exploration of expected cluster length

We further explored the relationship between the performance gap and the expected cluster length:

$$
E(L)=\frac{1}{1-p}
$$

The relationship between $\Delta$ and expected cluster length shows a nonlinear increasing trend, unlike the approximately linear relationship observed between $\Delta$ and $p$.

This suggests that, under the current sequence generation model, clustering probability provides a more direct description of the performance difference.

---

### 4. Summary and Future Work

This stage focuses on empirical observation and relationship discovery rather than theoretical proof.

The main conclusions are:

- Input structure has a significant impact on algorithm performance.
- Suffix Flip gains increasing advantages when the input sequence becomes more clustered.
- The performance gap between the two strategies shows a strong linear relationship with clustering probability.

Future work will focus on:

- Providing a theoretical explanation for the observed linear relationship.
- Testing whether the relationship remains valid under different sequence lengths.
- Exploring more adaptive suffix-based strategies.
- Investigating whether improved strategies can further exploit clustered structures.